# 04a – Artificial Neural Network (ANN) Regressor (PyTorch)
**Owner:** Member 4 (Deep Learning and Hybrid Engineer)

Builds, regularizes, and tunes a PyTorch feedforward ANN with ReLU activations, Dropout, L2 weight decay, Adam optimizer, MSE loss, and Early Stopping on validation data (never touching test data during tuning).

In [1]:
import copy
import random
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DOCS_DIR, FIGURES_DIR, MODELS_DIR, RANDOM_SEED, RESULTS_DIR
from src.evaluate import (
    compute_metrics,
    plot_parity,
    plot_residuals,
    save_model_run_metadata,
    save_predictions,
)
from src.features import build_preprocessor, get_xy

def set_seeds(seed: int = RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seeds(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

X_train_df, y_train_s, id_train = get_xy("train")
X_val_df, y_val_s, id_val = get_xy("val")
X_test_df, y_test_s, id_test = get_xy("test")

# Fit preprocessor ONLY on training data
preprocessor = build_preprocessor()
X_train = preprocessor.fit_transform(X_train_df).astype(np.float32)
X_val = preprocessor.transform(X_val_df).astype(np.float32)
X_test = preprocessor.transform(X_test_df).astype(np.float32)

y_train = y_train_s.to_numpy(dtype=np.float32)
y_val = y_val_s.to_numpy(dtype=np.float32)
y_test = y_test_s.to_numpy(dtype=np.float32)

print(f"Device: {DEVICE} | Input dim: {X_train.shape[1]}")

Device: cpu | Input dim: 193


In [2]:
class MOFRegressorANN(nn.Module):
    """Flexible MLP regressor with ReLU activations and Dropout."""
    def __init__(self, input_dim: int, hidden_dims: list[int], dropout: float = 0.1):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))  # Linear output layer
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_ann_model(config: dict, max_epochs: int = 180, patience: int = 20, seed: int = RANDOM_SEED):
    set_seeds(seed)
    model = MOFRegressorANN(
        input_dim=X_train.shape[1],
        hidden_dims=config["hidden_dims"],
        dropout=config["dropout"],
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"]
    )
    criterion = nn.MSELoss()

    train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
    g = torch.Generator()
    g.manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True, generator=g)

    X_val_t = torch.from_numpy(X_val).to(DEVICE)
    y_val_t = torch.from_numpy(y_val).to(DEVICE)

    train_losses, val_losses = [], []
    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    no_improve = 0

    t0 = time.perf_counter()
    for epoch in range(1, max_epochs + 1):
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
        train_mse = epoch_loss / len(X_train)

        model.eval()
        with torch.no_grad():
            val_preds = model(X_val_t)
            val_mse = float(criterion(val_preds, y_val_t).item())

        train_losses.append(train_mse)
        val_losses.append(val_mse)

        if val_mse < best_val_loss - 1e-5:
            best_val_loss = val_mse
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    train_time = time.perf_counter() - t0
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_t).cpu().numpy()
    val_metrics = compute_metrics(y_val, val_pred)
    return {
        "model": model,
        "config": config,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "epochs_run": len(train_losses),
        "best_epoch": best_epoch,
        "train_time": train_time,
        "val_metrics": val_metrics,
        "val_pred": val_pred,
    }

In [3]:
# Systematic hyperparameter tuning on Validation Set only
candidate_configs = [
    {"name": "baseline_64_32", "hidden_dims": [64, 32], "lr": 1e-3, "dropout": 0.1, "batch_size": 64, "weight_decay": 1e-4},
    {"name": "mlp_128_64", "hidden_dims": [128, 64], "lr": 1e-3, "dropout": 0.15, "batch_size": 64, "weight_decay": 1e-4},
    {"name": "mlp_128_64_32", "hidden_dims": [128, 64, 32], "lr": 1e-3, "dropout": 0.2, "batch_size": 64, "weight_decay": 1e-4},
    {"name": "mlp_256_128_64_lr5e4", "hidden_dims": [256, 128, 64], "lr": 5e-4, "dropout": 0.2, "batch_size": 32, "weight_decay": 1e-4},
    {"name": "mlp_128_64_32_lowdrop", "hidden_dims": [128, 64, 32], "lr": 5e-4, "dropout": 0.1, "batch_size": 32, "weight_decay": 5e-4},
    {"name": "mlp_64_32_16_lr5e4", "hidden_dims": [64, 32, 16], "lr": 5e-4, "dropout": 0.1, "batch_size": 32, "weight_decay": 1e-4},
]

runs = []
tuning_rows = []
for cfg in candidate_configs:
    res = train_ann_model(cfg)
    runs.append(res)
    m = res["val_metrics"]
    tuning_rows.append({
        "config": cfg["name"],
        "hidden_dims": str(cfg["hidden_dims"]),
        "lr": cfg["lr"],
        "dropout": cfg["dropout"],
        "batch_size": cfg["batch_size"],
        "weight_decay": cfg["weight_decay"],
        "epochs_run": res["epochs_run"],
        "best_epoch": res["best_epoch"],
        "val_r2": round(m["r2"], 4),
        "val_rmse": round(m["rmse"], 4),
        "val_mae": round(m["mae"], 4),
        "time_s": round(res["train_time"], 2),
    })

tuning_df = pd.DataFrame(tuning_rows).sort_values("val_rmse").reset_index(drop=True)
print(tuning_df.to_string(index=False))

               config    hidden_dims     lr  dropout  batch_size  weight_decay  epochs_run  best_epoch  val_r2  val_rmse  val_mae  time_s
mlp_128_64_32_lowdrop  [128, 64, 32] 0.0005     0.10          32        0.0005          90          70  0.6098    1.0486   0.6975   16.82
        mlp_128_64_32  [128, 64, 32] 0.0010     0.20          64        0.0001          90          70  0.5815    1.0859   0.7244   10.25
 mlp_256_128_64_lr5e4 [256, 128, 64] 0.0005     0.20          32        0.0001          69          49  0.5777    1.0908   0.7209   15.40
       baseline_64_32       [64, 32] 0.0010     0.10          64        0.0001          99          79  0.5772    1.0915   0.7631    8.88
           mlp_128_64      [128, 64] 0.0010     0.15          64        0.0001          56          36  0.5496    1.1265   0.7723    5.69
   mlp_64_32_16_lr5e4   [64, 32, 16] 0.0005     0.10          32        0.0001          65          45  0.5331    1.1470   0.8122   11.23


In [4]:
# Select best ANN based on Validation RMSE
best_run = min(runs, key=lambda r: r["val_metrics"]["rmse"])
best_model = best_run["model"]
best_cfg = best_run["config"]

# Plot training & validation loss curve
fig, ax = plt.subplots(figsize=(8, 4.8))
epochs_x = range(1, len(best_run["train_losses"]) + 1)
ax.plot(epochs_x, best_run["train_losses"], label="Training MSE Loss", linewidth=1.8)
ax.plot(epochs_x, best_run["val_losses"], label="Validation MSE Loss", linewidth=1.8)
ax.axvline(best_run["best_epoch"], color="red", linestyle="--", label=f"Best Epoch = {best_run['best_epoch']}")
ax.set_title(f"ANN Training & Validation Loss ({best_cfg['name']})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean Squared Error (MSE)")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "ann_loss_curve.png", dpi=200)
plt.close(fig)

# Evaluate finalized ANN on Test Set
best_model.eval()
with torch.no_grad():
    test_pred = best_model(torch.from_numpy(X_test).to(DEVICE)).cpu().numpy()

val_metrics = best_run["val_metrics"]
test_metrics = compute_metrics(y_test, test_pred)
print(f"Best ANN Config : {best_cfg}")
print(f"Validation      : {val_metrics}")
print(f"Test            : {test_metrics}")

# Save model, preprocessor, predictions, metadata, and tuning log
torch.save({"state_dict": best_model.state_dict(), "config": best_cfg, "input_dim": X_train.shape[1]}, MODELS_DIR / "ann_best.pt")
joblib.dump(preprocessor, MODELS_DIR / "ann_preprocessor.joblib")
save_predictions(id_val, y_val, best_run["val_pred"], "ann", "val")
save_predictions(id_test, y_test, test_pred, "ann", "test")
save_model_run_metadata(
    model_name="ann",
    display_name="ANN (PyTorch)",
    training_time_sec=best_run["train_time"],
    search_time_sec=sum(r["train_time"] for r in runs),
    extra={"best_config": best_cfg, "epochs_run": best_run["epochs_run"], "best_epoch": best_run["best_epoch"]},
)

plot_parity(y_test, test_pred, "ann", split="test")
plot_residuals(y_test, test_pred, "ann", split="test")

# Write docs/ann_tuning_log.md without extra dependencies
DOCS_DIR.mkdir(parents=True, exist_ok=True)
headers = list(tuning_df.columns)
md_lines = [
    "| " + " | ".join(headers) + " |\n",
    "| " + " | ".join([":---"] * len(headers)) + " |\n",
]
for _, row in tuning_df.iterrows():
    md_lines.append("| " + " | ".join(str(row[h]) for h in headers) + " |\n")

log_md = (
    "# ANN Hyperparameter Tuning Log\n"
    "**Owner:** Member 4 (Deep Learning and Hybrid Engineer)\n\n"
    "- **Framework:** PyTorch (Adam optimizer, MSE loss, ReLU hidden activations, linear output, Early Stopping patience=20)\n"
    f"- **Best Configuration:** `{best_cfg}` (Best Epoch: {best_run['best_epoch']} / {best_run['epochs_run']})\n\n"
    + "".join(md_lines)
)
(DOCS_DIR / "ann_tuning_log.md").write_text(log_md, encoding="utf-8")
print("Saved ANN model, loss curve, predictions, and docs/ann_tuning_log.md")

Best ANN Config : {'name': 'mlp_128_64_32_lowdrop', 'hidden_dims': [128, 64, 32], 'lr': 0.0005, 'dropout': 0.1, 'batch_size': 32, 'weight_decay': 0.0005}
Validation      : {'r2': 0.6097613395896282, 'rmse': 1.0485670089005512, 'mae': 0.6975323417258138}
Test            : {'r2': 0.5342919979319145, 'rmse': 0.9984620090886751, 'mae': 0.6917852953132242}


Saved ANN model, loss curve, predictions, and docs/ann_tuning_log.md
